In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

def summarize_file(filename, kpi_col):
    # load
    df = pd.read_csv(filename)
    # ensure timestamp column exists and convert to tz-aware datetime in Asia/Shanghai
    if 'timestamp' in df.columns:
        # convert epoch seconds -> UTC then convert to Asia/Shanghai
        df['_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        min_ts = df['_dt'].min()
        max_ts = df['_dt'].max()
        min_ts_s = min_ts.isoformat() if pd.notna(min_ts) else None
        max_ts_s = max_ts.isoformat() if pd.notna(max_ts) else None
    else:
        min_ts_s = None
        max_ts_s = None

    row_count = len(df)

    # distinct cmdb_id values (up to 50)
    if 'cmdb_id' in df.columns:
        cmdb_vals = df['cmdb_id'].dropna().astype(str).unique()
        cmdb_count = len(cmdb_vals)
        cmdb_sample = list(cmdb_vals[:50])
    else:
        cmdb_count = 0
        cmdb_sample = []

    # distinct KPI/feature/message values (up to 50)
    if kpi_col in df.columns:
        kpi_vals = df[kpi_col].dropna().astype(str).unique()
        kpi_count = len(kpi_vals)
        kpi_sample = list(kpi_vals[:50])
    else:
        kpi_count = 0
        kpi_sample = []

    # compact string representations (keep them readable but not extremely long)
    cmdb_display = ", ".join(cmdb_sample)
    kpi_display = ", ".join(kpi_sample)

    summary = {
        'file': filename,
        'row_count': row_count,
        'min_timestamp_ASIA/Shanghai': min_ts_s,
        'max_timestamp_ASIA/Shanghai': max_ts_s,
        'distinct_cmdb_id_count': cmdb_count,
        'cmdb_id_samples_up_to_50': cmdb_display,
        'distinct_name_count': kpi_count,
        'name_samples_up_to_50': kpi_display
    }
    # keep full df in memory in case of follow-up (variable reuse)
    return pd.DataFrame([summary]), df

# Summarize each file
metric_summary, metric_df = summarize_file('metric.csv', 'kpi_name')
trace_summary, trace_df = summarize_file('trace.csv', 'trace_name')
log_summary, log_df = summarize_file('log.csv', 'log_name')
error_summary, error_df = summarize_file('error_logs.csv', 'message')

# Combine summaries into a compact table (one row per file)
summary_df = pd.concat([metric_summary, trace_summary, log_summary, error_summary], ignore_index=True)

# Display the compact summary table (4 rows). This is the final displayed variable.
summary_df

```
Out[1]:
```
```python
# Summarize the previously computed file summaries in plain English.
# Reuse the existing `summary_df` variable from the kernel state.

rows = []
for _, r in summary_df.iterrows():
    file = r['file']
    rc = int(r['row_count'])
    tmin = r['min_timestamp_ASIA/Shanghai']
    tmax = r['max_timestamp_ASIA/Shanghai']
    cmdb_count = int(r['distinct_cmdb_id_count'])
    # pick up to 5 cmdb samples for a concise summary
    cmdb_samples = [s.strip() for s in str(r['cmdb_id_samples_up_to_50']).split(',') if s.strip()][:5]
    cmdb_sample_str = ", ".join(cmdb_samples) if cmdb_samples else "None"
    name_count = int(r['distinct_name_count'])
    name_samples = [s.strip() for s in str(r['name_samples_up_to_50']).split(',') if s.strip()][:5]
    name_sample_str = ", ".join(name_samples) if name_samples else "None"
    rows.append(f"{file}: {rc} rows; time range {tmin} to {tmax}; {cmdb_count} distinct cmdb_id (examples: {cmdb_sample_str}); {name_count} distinct names/features (examples: {name_sample_str})")

summary = "File summaries:\n" + "\n".join(rows)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

file  row_count min_timestamp_ASIA/Shanghai max_timestamp_ASIA/Shanghai  distinct_cmdb_id_count                           cmdb_id_samples_up_to_50  distinct_name_count                              name_samples_up_to_50
0      metric.csv     148332   2022-03-21T11:00:00+08:00   2022-03-21T11:29:00+08:00                      59  adservice, adservice-0, adservice-1, adservice...                 1228  app.grpc.count, app.grpc.mrt, app.grpc.rr, app...
1       trace.csv      34304   2022-03-21T11:00:00+08:00   2022-03-21T11:29:00+08:00                      40  adservice-0, adservice-1, adservice-2, adservi...                  200  trace.from_frontend-0.duration_mean, trace.fro...
2         log.csv       1758   2022-03-21T11:00:00+08:00   2022-03-21T11:29:00+08:00                      31  adservice-0, adservice-1, adservice-2, cartser...                    2                     log.error_count, log.row_count
3  error_logs.csv       1708   2022-03-21T11:00:00+08:00   2022-03-21T11:29:58+08:00                       3                 frontend-1, frontend-2, frontend-0                   47  severity: warning, message: failed to retrieve...```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_df is loaded (reuse if available in kernel)
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure timezone-aware datetime column exists
if '_dt' not in metric_df.columns:
    metric_df['_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure column is tz-aware and in Asia/Shanghai
    metric_df['_dt'] = pd.to_datetime(metric_df['_dt']).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, kpi_name) using full series (no filtering)
p95_df = metric_df.groupby(['cmdb_id', 'kpi_name'], sort=False)['value'].quantile(0.95).reset_index()
p95_df = p95_df.rename(columns={'value': 'p95_threshold'})

# 2) Filter metric rows to the incident window (inclusive)
start = pd.Timestamp('2022-03-21T11:00:00', tz=tz)
end = pd.Timestamp('2022-03-21T11:30:00', tz=tz)
window_df = metric_df[(metric_df['_dt'] >= start) & (metric_df['_dt'] <= end)].copy()

# 3) Merge P95 into the windowed data (thresholds computed on full series)
window_with_p95 = window_df.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomalies where value >= global P95
window_with_p95['is_anom'] = window_with_p95['value'] >= window_with_p95['p95_threshold']

# Group and compute requested metrics per (cmdb_id, kpi_name)
g = window_with_p95.groupby(['cmdb_id', 'kpi_name', 'p95_threshold'], sort=False)

anomaly_count = g['is_anom'].sum().astype(int)
earliest_anomaly_dt = g.apply(lambda df: df.loc[df['is_anom'], '_dt'].min())
max_value = g['value'].max()

# Assemble full grouped results (store in kernel variable)
grouped_all = pd.concat([anomaly_count, earliest_anomaly_dt, max_value], axis=1)
grouped_all.columns = ['anomaly_count', 'earliest_anomaly_dt', 'max_value_in_window']
grouped_all = grouped_all.reset_index()

# Convert earliest_anomaly_dt to ISO strings (keep None for groups without anomalies)
grouped_all['earliest_anomaly_timestamp_ASIA/Shanghai'] = grouped_all['earliest_anomaly_dt'].apply(
    lambda x: x.isoformat() if pd.notna(x) else None
)

# 4) Filter to anomaly_count > 0 and show top 20 by anomaly_count desc
result_top20 = (
    grouped_all[grouped_all['anomaly_count'] > 0]
    .sort_values(by='anomaly_count', ascending=False)
    .loc[:, ['cmdb_id', 'kpi_name', 'p95_threshold', 'anomaly_count',
             'earliest_anomaly_timestamp_ASIA/Shanghai', 'max_value_in_window']]
    .head(20)
)

# Keep grouped_all in kernel (full intermediate results) and display compact top-20 result.
grouped_all, result_top20

```
Out[2]:
```
```python
# Build a concise plain-English summary using the previously computed grouped results.
# Reuse `grouped_all` and `result_top20` from the kernel state if available.

import pandas as pd

try:
    ga = grouped_all
    top20 = result_top20
except NameError:
    # If grouped results are not available, raise to indicate prior step needed.
    raise RuntimeError("Grouped intermediate results not found in kernel. Please run the previous metric analysis step first.")

# Incident window (as used previously)
start = "2022-03-21T11:00:00+08:00"
end = "2022-03-21T11:30:00+08:00"

# Totals
total_pairs_with_anom = int((ga['anomaly_count'] > 0).sum())
total_examined_pairs = int(ga.shape[0])

# Max anomaly count and examples
max_anom = int(ga['anomaly_count'].max())
count_at_max = int((ga['anomaly_count'] == max_anom).sum())

# Prepare up to 5 example rows at the max anomaly_count
examples_max_df = ga[ga['anomaly_count'] == max_anom].loc[:, 
    ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp_ASIA/Shanghai', 'max_value_in_window']
].head(5)

examples_list = []
for _, r in examples_max_df.iterrows():
    examples_list.append(f"{r['cmdb_id']} | {r['kpi_name']} | anomalies={int(r['anomaly_count'])} | earliest={r['earliest_anomaly_timestamp_ASIA/Shanghai']} | max={r['max_value_in_window']}")
examples_text = "; ".join(examples_list) if examples_list else "None"

# Top 3 (cmdb_id, kpi_name) by anomaly_count for quick insight (from result_top20)
top3 = top20.head(3)
top3_lines = []
for _, r in top3.iterrows():
    top3_lines.append(f"{r['cmdb_id']} | {r['kpi_name']} | anomalies={int(r['anomaly_count'])} | earliest={r['earliest_anomaly_timestamp_ASIA/Shanghai']} | max={r['max_value_in_window']}")
top3_text = " ; ".join(top3_lines) if top3_lines else "None"

summary = (
    f"Metric anomaly summary for window {start} to {end} (P95 computed on full series):\n"
    f"- Examined (cmdb_id, kpi_name) pairs in window: {total_examined_pairs}\n"
    f"- Pairs with >=1 anomaly (value >= global P95): {total_pairs_with_anom}\n"
    f"- Highest anomaly_count observed: {max_anom} (affects {count_at_max} pair(s)). Examples at that max: {examples_text}\n"
    f"- Top 3 pairs by anomaly_count: {top3_text}\n"
    f"(Full grouped results are stored in the kernel variable `grouped_all`; top-20 filtered table is `result_top20`.)"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                cmdb_id                                           kpi_name  p95_threshold  anomaly_count       earliest_anomaly_dt  max_value_in_window earliest_anomaly_timestamp_ASIA/Shanghai
0             adservice                                     app.grpc.count     238.000000              3 2022-03-21 11:10:00+08:00           243.000000                2022-03-21T11:10:00+08:00
1             adservice                                       app.grpc.mrt       0.002839              2 2022-03-21 11:02:00+08:00             0.003469                2022-03-21T11:02:00+08:00
2             adservice                                        app.grpc.rr     100.000000             30 2022-03-21 11:00:00+08:00           100.000000                2022-03-21T11:00:00+08:00
3             adservice                                        app.grpc.sr      77.412921              2 2022-03-21 11:07:00+08:00            78.289474                2022-03-21T11:07:00+08:00
4             adservice                                     app.http.count      24.000000             30 2022-03-21 11:00:00+08:00            24.000000                2022-03-21T11:00:00+08:00
...                 ...                                                ...            ...            ...                       ...                  ...                                      ...
4953  checkoutservice-2  mesh.destination.frontend.checkoutservice.isti...       0.000000             21 2022-03-21 11:09:00+08:00             0.000000                2022-03-21T11:09:00+08:00
4954  checkoutservice-2  mesh.source.checkoutservice.productcatalogserv...       0.000000             21 2022-03-21 11:09:00+08:00             0.000000                2022-03-21T11:09:00+08:00
4955  checkoutservice-2  mesh.source.checkoutservice.productcatalogserv...       0.000000             21 2022-03-21 11:09:00+08:00             0.000000                2022-03-21T11:09:00+08:00
4956  checkoutservice-2  mesh.source.checkoutservice.productcatalogserv...       0.000000             21 2022-03-21 11:09:00+08:00             0.000000                2022-03-21T11:09:00+08:00
4957  checkoutservice-2  mesh.source.checkoutservice.productcatalogserv...       0.000000             21 2022-03-21 11:09:00+08:00             0.000000                2022-03-21T11:09:00+08:00

[4958 rows x 7 columns],                  cmdb_id                                           kpi_name  p95_threshold  anomaly_count earliest_anomaly_timestamp_ASIA/Shanghai  max_value_in_window
4891  shippingservice2-0  container.node-5.container_network_receive_err...   0.000000e+00             30                2022-03-21T11:00:00+08:00         0.000000e+00
4893  shippingservice2-0  container.node-5.container_network_receive_pac...   0.000000e+00             30                2022-03-21T11:00:00+08:00         0.000000e+00
4895  shippingservice2-0  container.node-5.container_network_transmit_er...   0.000000e+00             30                2022-03-21T11:00:00+08:00         0.000000e+00
4881  shippingservice2-0  container.node-5.container_memory_failures.con...   0.000000e+00             30                2022-03-21T11:00:00+08:00         0.000000e+00
4883  shippingservice2-0  container.node-5.container_memory_failures.hie...   0.000000e+00             30                2022-03-21T11:00:00+08:00         0.000000e+00
4885  shippingservice2-0     container.node-5.container_memory_max_usage_MB   1.280117e+02             30                2022-03-21T11:00:00+08:00         1.280117e+02
4887  shippingservice2-0             container.node-5.container_memory_swap   0.000000e+00             30                2022-03-21T11:00:00+08:00         0.000000e+00
4872  shippingservice2-0  container.node-5.container_fs_write_seconds./d...   0.000000e+00             30                2022-03-21T11:00:00+08:00         0.000000e+00
4873  shippingservice2-0      container.node-5.container_fs_writes./dev/vda   0.000000e+00             30                2022-03-21T11:00:00+08:00         0.000000e+00
4874  shippingservice2-0     container.node-5.container_fs_writes./dev/vda1   0.000000e+00             30                2022-03-21T11:00:00+08:00         0.000000e+00
4875  shippingservice2-0   container.node-5.container_fs_writes_MB./dev/vda   0.000000e+00             30                2022-03-21T11:00:00+08:00         0.000000e+00
4876  shippingservice2-0  container.node-5.container_fs_writes_merged./d...   0.000000e+00             30                2022-03-21T11:00:00+08:00         0.000000e+00
4878  shippingservice2-0            container.node-5.container_memory_cache   1.803059e+07             30                2022-03-21T11:00:00+08:00         1.803059e+07
4879  shippingservice2-0          container.node-5.container_memory_failcnt   2.397029e+06             30                2022-03-21T11:00:00+08:00         2.397029e+06
4864  shippingservice2-0  container.node-5.container_fs_read_seconds./de...   0.000000e+00             30                2022-03-21T11:00:00+08:00         0.000000e+00
4865  shippingservice2-0       container.node-5.container_fs_reads./dev/vda   0.000000e+00             30                2022-03-21T11:00:00+08:00         0.000000e+00
4866  shippingservice2-0      container.node-5.container_fs_reads./dev/vda1   0.000000e+00             30                2022-03-21T11:00:00+08:00         0.000000e+00
4867  shippingservice2-0    container.node-5.container_fs_reads_MB./dev/vda   0.000000e+00             30                2022-03-21T11:00:00+08:00         0.000000e+00
4868  shippingservice2-0  container.node-5.container_fs_reads_merged./de...   0.000000e+00             30                2022-03-21T11:00:00+08:00         0.000000e+00
4869  shippingservice2-0  container.node-5.container_fs_sector_reads./de...   0.000000e+00             30                2022-03-21T11:00:00+08:00         0.000000e+00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing grouped_all from the kernel
try:
    ga = grouped_all
except NameError:
    raise RuntimeError("Kernel variable `grouped_all` not found. Please run the previous analysis step first.")

# Work only with rows that have anomalies (anomaly_count > 0)
ga_anom = ga[ga['anomaly_count'] > 0].copy()

# Ensure earliest_anomaly_dt is datetime (tz-aware). If it's string, try to parse.
if 'earliest_anomaly_dt' in ga_anom.columns:
    ga_anom['earliest_anomaly_dt'] = pd.to_datetime(ga_anom['earliest_anomaly_dt'], utc=True).dt.tz_convert(tz)
else:
    # fallback: try to use the timestamp string column if present
    if 'earliest_anomaly_timestamp_ASIA/Shanghai' in ga_anom.columns:
        ga_anom['earliest_anomaly_dt'] = pd.to_datetime(ga_anom['earliest_anomaly_timestamp_ASIA/Shanghai']).dt.tz_localize(tz, ambiguous='NaT', nonexistent='NaT')

# Aggregate by cmdb_id
component_anom_summary = (
    ga_anom
    .groupby('cmdb_id', sort=False)
    .agg(
        distinct_anomalous_kpi_count = ('kpi_name', lambda x: x.astype(str).nunique()),
        sum_anomaly_points = ('anomaly_count', 'sum'),
        earliest_anomaly_dt = ('earliest_anomaly_dt', 'min'),
        example_kpi_samples = ('kpi_name', lambda x: ", ".join(pd.Series(x.astype(str)).unique()[:5]))
    )
    .reset_index()
)

# Convert earliest_anomaly_dt to ISO string column as required for display
component_anom_summary['earliest_anomaly_timestamp_ASIA/Shanghai'] = component_anom_summary['earliest_anomaly_dt'].apply(
    lambda x: x.isoformat() if pd.notna(x) else None
)

# Order by distinct_anomalous_kpi_count desc, then sum_anomaly_points desc
component_anom_summary = component_anom_summary.sort_values(
    by=['distinct_anomalous_kpi_count', 'sum_anomaly_points'],
    ascending=[False, False]
).reset_index(drop=True)

# Select and order the final display columns
component_anom_summary = component_anom_summary.loc[:, [
    'cmdb_id',
    'distinct_anomalous_kpi_count',
    'sum_anomaly_points',
    'earliest_anomaly_timestamp_ASIA/Shanghai',
    'example_kpi_samples'
]]

# Keep full aggregated table in kernel variable `component_anom_summary`
# Display only top 20 rows as requested
component_anom_summary.head(20)

```
Out[3]:
```
Summary of component-level anomaly aggregation (based on anomalies where metric value >= global P95):

- Overall pattern: Many components show multiple anomalous KPIs starting at the beginning of the incident window. Earliest anomalies are typically at 2022-03-21T11:00:00+08:00, indicating the issue began at the window start.

- Top affected components (top 5 by distinct anomalous KPI count):
  1. adservice — 341 distinct anomalous KPIs, sum of anomaly points = 8,908, earliest anomaly 2022-03-21T11:00:00+08:00 (example KPIs: app.grpc.count, app.grpc.mrt, app.grpc.rr, ...)
  2. adservice2 — 333 distinct anomalous KPIs, sum = 8,315, earliest 2022-03-21T11:00:00+08:00 (example KPIs: runtime.java_lang_ClassLoading_LoadedClassCount, ...)
  3. frontend2-0 — 176 distinct anomalous KPIs, sum = 2,933, earliest 2022-03-21T11:00:00+08:00 (example KPIs: container.node-6.container_cpu_cfs_periods, ...)
  4. frontend-1 — 168 distinct anomalous KPIs, sum = 2,724, earliest 2022-03-21T11:00:00+08:00
  5. frontend-0 — 168 distinct anomalous KPIs, sum = 2,660, earliest 2022-03-21T11:00:00+08:00

- Additional observation: Many example KPIs are container/node-level metrics (e.g., container.node-5.container_cpu_cfs_periods, container memory/cpu/fs metrics), suggesting resource- or node-related abnormalities affecting multiple services/pods.

- Data availability: The full aggregated table is stored in the kernel variable component_anom_summary if you want to inspect more rows or details.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  distinct_anomalous_kpi_count  sum_anomaly_points earliest_anomaly_timestamp_ASIA/Shanghai                                example_kpi_samples
0                  adservice                           341                8908                2022-03-21T11:00:00+08:00  app.grpc.count, app.grpc.mrt, app.grpc.rr, app...
1                 adservice2                           333                8315                2022-03-21T11:00:00+08:00  runtime.java_lang_ClassLoading_LoadedClassCoun...
2                frontend2-0                           176                2933                2022-03-21T11:00:00+08:00  container.node-6.container_cpu_cfs_periods, co...
3                 frontend-1                           168                2724                2022-03-21T11:00:00+08:00  container.node-5.container_cpu_cfs_periods, co...
4                 frontend-0                           168                2660                2022-03-21T11:00:00+08:00  container.node-5.container_cpu_cfs_periods, co...
5                 frontend-2                           158                2459                2022-03-21T11:00:00+08:00  container.node-5.container_cpu_cfs_periods, co...
6          checkoutservice-0                           130                2018                2022-03-21T11:00:00+08:00  container.node-5.container_cpu_cfs_periods, co...
7          checkoutservice-2                           130                1924                2022-03-21T11:00:00+08:00  container.node-5.container_cpu_cfs_periods, co...
8          checkoutservice-1                           118                1647                2022-03-21T11:00:00+08:00  container.node-5.container_cpu_cfs_periods, co...
9         checkoutservice2-0                           118                1635                2022-03-21T11:00:00+08:00  container.node-5.container_cpu_cfs_periods, co...
10   productcatalogservice-2                           116                2207                2022-03-21T11:00:00+08:00  container.node-5.container_cpu_cfs_periods, co...
11   productcatalogservice-0                           116                2154                2022-03-21T11:00:00+08:00  container.node-5.container_cpu_cfs_periods, co...
12   productcatalogservice-1                           116                2143                2022-03-21T11:00:00+08:00  container.node-5.container_cpu_cfs_periods, co...
13              adservice2-0                           102                2088                2022-03-21T11:00:00+08:00  container.node-6.container_cpu_cfs_periods, co...
14             cartservice-1                           100                2173                2022-03-21T11:00:00+08:00  container.node-5.container_cpu_cfs_periods, co...
15             cartservice-0                            96                2052                2022-03-21T11:00:00+08:00  container.node-5.container_cpu_cfs_periods, co...
16            cartservice2-0                            96                2010                2022-03-21T11:00:00+08:00  container.node-6.container_cpu_cfs_periods, co...
17  productcatalogservice2-0                            96                1872                2022-03-21T11:00:00+08:00  container.node-6.container_cpu_cfs_periods, co...
18             cartservice-2                            88                1747                2022-03-21T11:00:00+08:00  container.node-5.container_cpu_cfs_periods, co...
19               adservice-2                            86                1885                2022-03-21T11:00:00+08:00  container.node-5.container_cpu_cfs_periods, co...```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing trace_df if available, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure timezone-aware datetime column _dt exists and is in Asia/Shanghai
if '_dt' not in trace_df.columns:
    trace_df['_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    trace_df['_dt'] = pd.to_datetime(trace_df['_dt']).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, trace_name) using full series
p95_trace = trace_df.groupby(['cmdb_id', 'trace_name'], sort=False)['value'].quantile(0.95).reset_index()
p95_trace = p95_trace.rename(columns={'value': 'p95_threshold'})

# 2) Filter trace rows to the incident window (inclusive)
start = pd.Timestamp('2022-03-21T11:00:00', tz=tz)
end = pd.Timestamp('2022-03-21T11:30:00', tz=tz)
trace_window = trace_df[(trace_df['_dt'] >= start) & (trace_df['_dt'] <= end)].copy()

# 3) Merge p95 into windowed data and mark anomalies
trace_window = trace_window.merge(p95_trace, on=['cmdb_id', 'trace_name'], how='left')
trace_window['is_anom'] = trace_window['value'] >= trace_window['p95_threshold']

# Group and compute requested metrics per (cmdb_id, trace_name)
g = trace_window.groupby(['cmdb_id', 'trace_name', 'p95_threshold'], sort=False)

anomaly_count = g['is_anom'].sum().astype(int)
earliest_anom_dt = g.apply(lambda df: df.loc[df['is_anom'], '_dt'].min())
max_value = g['value'].max()

grouped_trace_all = pd.concat([anomaly_count, earliest_anom_dt, max_value], axis=1)
grouped_trace_all.columns = ['anomaly_count', 'earliest_anomaly_dt', 'max_value_in_window']
grouped_trace_all = grouped_trace_all.reset_index()

# Convert earliest_anomaly_dt to ISO string (keep None for groups without anomalies)
grouped_trace_all['earliest_anomaly_timestamp_ASIA/Shanghai'] = grouped_trace_all['earliest_anomaly_dt'].apply(
    lambda x: x.isoformat() if pd.notna(x) else None
)

# 4) Filter to anomaly_count > 0 and get top 20 by anomaly_count desc
result_trace_top20 = (
    grouped_trace_all[grouped_trace_all['anomaly_count'] > 0]
    .sort_values(by='anomaly_count', ascending=False)
    .loc[:, ['cmdb_id', 'trace_name', 'p95_threshold', 'anomaly_count',
             'earliest_anomaly_timestamp_ASIA/Shanghai', 'max_value_in_window']]
    .head(20)
    .reset_index(drop=True)
)

# Store grouped_trace_all and result_trace_top20 in kernel (already assigned)
# Display a compact indicator for the full grouped results and the top-20 table
grouped_trace_all.shape, result_trace_top20

```
Out[4]:
```
```python
# Summarize trace anomaly results in plain English using existing kernel variables
# Reuse grouped_trace_all and result_trace_top20 from the kernel state.

try:
    gta = grouped_trace_all
    top20 = result_trace_top20
except NameError:
    raise RuntimeError("Kernel variables `grouped_trace_all` or `result_trace_top20` not found. Run the trace analysis step first.")

total_pairs = int(gta.shape[0])
pairs_with_anom = int((gta['anomaly_count'] > 0).sum())
max_anom = int(gta['anomaly_count'].max())
count_at_max = int((gta['anomaly_count'] == max_anom).sum())

# examples for highest-anomaly pairs (up to 3)
examples_max = gta[gta['anomaly_count'] == max_anom].loc[:, 
    ['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_timestamp_ASIA/Shanghai', 'max_value_in_window']
].head(3)

examples_text = "; ".join(
    f"{row.cmdb_id} | {row.trace_name} | anomalies={int(row.anomaly_count)} | earliest={row.earliest_anomaly_timestamp_ASIA/Shanghai} | max={row.max_value_in_window}"
    for _, row in examples_max.iterrows()
) if not examples_max.empty else "None"

# top 3 from result_trace_top20 for quick insight
top3 = top20.head(3)
top3_text = "; ".join(
    f"{row.cmdb_id} | {row.trace_name} | anomalies={int(row.anomaly_count)} | earliest={row.earliest_anomaly_timestamp_ASIA/Shanghai} | max={row.max_value_in_window}"
    for _, row in top3.iterrows()
) if not top3.empty else "None"

summary = (
    f"Trace anomaly summary for 2022-03-21T11:00:00+08:00 to 2022-03-21T11:30:00+08:00 (P95 computed on full series):\n"
    f"- Total (cmdb_id, trace_name) pairs examined: {total_pairs}\n"
    f"- Pairs with >=1 anomaly (value >= global P95): {pairs_with_anom}\n"
    f"- Highest anomaly_count observed: {max_anom} (affects {count_at_max} pair(s)). Examples at that max: {examples_text}\n"
    f"- Top 3 pairs by anomaly_count (from top-20): {top3_text}\n"
    f"- Note: many trace P95 thresholds are 0.0, which results in the whole incident window being flagged when values are >= 0 (see top results where anomaly_count = 30 and earliest anomaly = 2022-03-21T11:00:00+08:00).\n"
    f"(Full grouped trace results are in kernel variable `grouped_trace_all`; top-20 results are in `result_trace_top20`.)"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((1232, 7),                cmdb_id                                   trace_name  p95_threshold  anomaly_count earliest_anomaly_timestamp_ASIA/Shanghai  max_value_in_window
0          adservice-1             trace.from_frontend-0.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000
1   shippingservice2-0            trace.from_frontend2-0.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000
2          adservice-0             trace.from_frontend-2.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000
3          adservice-1             trace.from_frontend-2.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000
4          adservice-0             trace.from_frontend-0.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000
5          adservice-1             trace.from_frontend-1.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000
6          adservice-2             trace.from_frontend-1.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000
7           frontend-0          trace.to_cartservice-0.duration_p95            0.0             30                2022-03-21T11:00:00+08:00                0.000
8           frontend-0            trace.to_cartservice-0.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000
9           frontend-0  trace.to_recommendationservice-1.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000
10          frontend-0  trace.to_recommendationservice-2.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000
11          frontend-0            trace.to_cartservice-1.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000
12          frontend-0          trace.to_cartservice-2.duration_p95            0.0             30                2022-03-21T11:00:00+08:00                0.001
13          frontend-0            trace.to_cartservice-2.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000
14          frontend-0              trace.to_adservice-2.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000
15          frontend-0        trace.to_shippingservice-1.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000
16          frontend-0        trace.to_currencyservice-0.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000
17          frontend-1              trace.to_adservice-0.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000
18          frontend-0        trace.to_shippingservice-2.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000
19          frontend-1                   trace.from_root.error_rate            0.0             30                2022-03-21T11:00:00+08:00                0.000)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse or load error_logs and log dataframes
try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')

try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

# Ensure timezone-aware datetime columns for both datasets
for df, name in [(error_df, 'error_df'), (log_df, 'log_df')]:
    if '_dt' not in df.columns:
        df['_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df['_dt'] = pd.to_datetime(df['_dt']).dt.tz_convert(tz)

# 1) error_logs.csv aggregation per cmdb_id
# total_message_count, earliest and latest timestamps, up to 5 sample messages
err_grp = error_df.groupby('cmdb_id', sort=False)

error_log_summary = err_grp.agg(
    total_message_count = ('message', 'count'),
    earliest_message_dt = ('_dt', 'min'),
    latest_message_dt = ('_dt', 'max'),
    # collect up to 5 unique sample messages (preserve order)
    sample_messages = ('message', lambda s: ", ".join(pd.unique(s.astype(str))[:5]))
).reset_index()

# Convert datetimes to ISO strings for compact display
error_log_summary['earliest_message_timestamp_ASIA/Shanghai'] = error_log_summary['earliest_message_dt'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
error_log_summary['latest_message_timestamp_ASIA/Shanghai'] = error_log_summary['latest_message_dt'].apply(lambda x: x.isoformat() if pd.notna(x) else None)

# Keep full aggregated table in kernel variable `error_log_summary`
# Sort by total_message_count desc and prepare compact display version (top 20)
error_log_summary = error_log_summary.sort_values(by='total_message_count', ascending=False).reset_index(drop=True)
error_log_summary_display = error_log_summary.loc[:, [
    'cmdb_id',
    'total_message_count',
    'earliest_message_timestamp_ASIA/Shanghai',
    'latest_message_timestamp_ASIA/Shanghai',
    'sample_messages'
]].head(20)

# 2) log.csv anomaly detection per (cmdb_id, log_name)
# 2a) compute global P95 per pair using full series
p95_logs = log_df.groupby(['cmdb_id', 'log_name'], sort=False)['value'].quantile(0.95).reset_index()
p95_logs = p95_logs.rename(columns={'value': 'p95_threshold'})

# 2b) filter log rows to incident window (inclusive)
start = pd.Timestamp('2022-03-21T11:00:00', tz=tz)
end = pd.Timestamp('2022-03-21T11:30:00', tz=tz)
log_window = log_df[(log_df['_dt'] >= start) & (log_df['_dt'] <= end)].copy()

# 2c) merge p95 thresholds into windowed data
log_window = log_window.merge(p95_logs, on=['cmdb_id', 'log_name'], how='left')

# mark anomalies: value >= p95_threshold AND p95_threshold not NaN
log_window['is_anom'] = (~log_window['p95_threshold'].isna()) & (log_window['value'] >= log_window['p95_threshold'])

# group and compute required metrics per pair
g = log_window.groupby(['cmdb_id', 'log_name', 'p95_threshold'], sort=False)

anomaly_count = g['is_anom'].sum().astype(int)
earliest_anom_dt = g.apply(lambda df: df.loc[df['is_anom'], '_dt'].min())
max_value = g['value'].max()

grouped = pd.concat([anomaly_count, earliest_anom_dt, max_value], axis=1)
grouped.columns = ['anomaly_count', 'earliest_anomaly_dt', 'max_value_in_window']
grouped = grouped.reset_index()

# Add ISO string timestamp column
grouped['earliest_anomaly_timestamp_ASIA/Shanghai'] = grouped['earliest_anomaly_dt'].apply(lambda x: x.isoformat() if pd.notna(x) else None)

# Store full result in kernel variable `log_metric_anom`
log_metric_anom = grouped.copy()

# Filter to anomaly_count > 0 and prepare top-20 sorted by anomaly_count desc
log_metric_anom_top20 = (
    log_metric_anom[log_metric_anom['anomaly_count'] > 0]
    .sort_values(by=['anomaly_count', 'max_value_in_window'], ascending=[False, False])
    .loc[:, ['cmdb_id', 'log_name', 'p95_threshold', 'anomaly_count', 'earliest_anomaly_timestamp_ASIA/Shanghai', 'max_value_in_window']]
    .head(20)
    .reset_index(drop=True)
)

# Display compact outputs: top 20 error_log_summary and top 20 log anomalies
error_log_summary_display, log_metric_anom_top20

```
Out[5]:
```
```python
# Summarize findings from error_logs and log anomaly analysis in plain English.
# Reuse kernel variables produced earlier: error_log_summary (full aggregated error logs)
# and log_metric_anom / log_metric_anom_top20 (log anomalies). Display a compact summary string.

import pandas as pd

# attempt to reuse available variables from kernel state
try:
    err_full = error_log_summary
except NameError:
    # fallback to displayed version if full not present
    try:
        err_full = error_log_summary_display
    except NameError:
        err_full = None

# attempt to get log anomalies top table
try:
    log_top = log_metric_anom_top20
except NameError:
    try:
        # if top20 not present, use full and filter
        log_full = log_metric_anom
        log_top = (log_full[log_full['anomaly_count'] > 0]
                   .sort_values(by='anomaly_count', ascending=False)
                   .loc[:, ['cmdb_id','log_name','p95_threshold','anomaly_count','earliest_anomaly_timestamp_ASIA/Shanghai','max_value_in_window']]
                   .head(20))
    except NameError:
        log_top = None

lines = []
lines.append("Compact evidence summary from error_logs.csv and log.csv:")

# Summarize error logs
if isinstance(err_full, pd.DataFrame) and not err_full.empty:
    top_err = err_full.sort_values('total_message_count', ascending=False).head(3)
    lines.append("- Error logs (top components by message count):")
    for _, r in top_err.iterrows():
        cm = r['cmdb_id']
        cnt = int(r['total_message_count'])
        tmin = r.get('earliest_message_timestamp_ASIA/Shanghai', None)
        tmax = r.get('latest_message_timestamp_ASIA/Shanghai', None)
        samples = r.get('sample_messages', '')
        lines.append(f"  • {cm}: {cnt} messages, time {tmin} → {tmax}; sample messages: {samples.split(',')[0:2]}")
    lines.append("  Note: Frontend pods (frontend-2, frontend-1, frontend-0) have the highest message volumes (~618, 550, 540) with earliest messages at ~2022-03-21T11:00:00+08:00.")
else:
    lines.append("- Error log summary not found in kernel.")

# Summarize log anomalies
if isinstance(log_top, pd.DataFrame) and not log_top.empty:
    lines.append("- Log metric anomalies (top log (cmdb_id, log_name) pairs):")
    # show up to 5 examples
    for _, r in log_top.head(5).iterrows():
        cm = r['cmdb_id']
        ln = r['log_name']
        p95 = r['p95_threshold']
        ac = int(r['anomaly_count'])
        et = r['earliest_anomaly_timestamp_ASIA/Shanghai']
        mv = r['max_value_in_window']
        lines.append(f"  • {cm} | {ln}: p95={p95} → anomalies={ac}, earliest={et}, max_in_window={mv}")
    lines.append("  Observation: Many log.error_count p95 thresholds are 0.0, producing anomaly_count=30 across pods (earliest 2022-03-21T11:00:00+08:00). This means the P95 is trivial (0) and the entire incident window is flagged.")
else:
    lines.append("- Log anomaly summary not found in kernel.")

lines.append("- Overall interpretation: Frontend pods show high error message volumes starting at the window start (11:00+08). Log metric anomalies are widespread but driven by zero p95 thresholds, so treat those anomaly flags cautiously and inspect raw messages and metrics for real errors.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  total_message_count earliest_message_timestamp_ASIA/Shanghai latest_message_timestamp_ASIA/Shanghai                                    sample_messages
0  frontend-2                  618                2022-03-21T11:00:00+08:00              2022-03-21T11:29:53+08:00  severity: warning, message: failed to retrieve...
1  frontend-1                  550                2022-03-21T11:00:00+08:00              2022-03-21T11:29:56+08:00  severity: warning, message: failed to retrieve...
2  frontend-0                  540                2022-03-21T11:00:01+08:00              2022-03-21T11:29:58+08:00  severity: warning, message: failed to retrieve...,                     cmdb_id         log_name  p95_threshold  anomaly_count earliest_anomaly_timestamp_ASIA/Shanghai  max_value_in_window
0               adservice-0  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
1               adservice-1  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
2               adservice-2  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
3             cartservice-0  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
4             cartservice-1  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
5             cartservice-2  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
6         checkoutservice-0  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
7         checkoutservice-1  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
8         checkoutservice-2  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
9         currencyservice-0  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
10        currencyservice-1  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
11        currencyservice-2  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
12  productcatalogservice-0  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
13  productcatalogservice-1  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
14  productcatalogservice-2  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
15  recommendationservice-0  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
16  recommendationservice-1  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
17  recommendationservice-2  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
18        shippingservice-0  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0
19        shippingservice-1  log.error_count            0.0             30                2022-03-21T11:00:00+08:00                  0.0)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse grouped_all from kernel
try:
    ga = grouped_all
except NameError:
    raise RuntimeError("Kernel variable `grouped_all` not found. Please run the metric grouping step first.")

# Ensure required columns exist and types are correct
if 'kpi_name' not in ga.columns or 'anomaly_count' not in ga.columns:
    raise RuntimeError("`grouped_all` must contain 'kpi_name' and 'anomaly_count' columns.")

# Ensure anomaly_count numeric
ga['anomaly_count'] = pd.to_numeric(ga['anomaly_count'], errors='coerce').fillna(0).astype(int)

# Ensure we have a datetime column for earliest anomaly; prefer 'earliest_anomaly_dt' if present
if 'earliest_anomaly_dt' in ga.columns:
    ga['earliest_anomaly_dt'] = pd.to_datetime(ga['earliest_anomaly_dt'], utc=True).dt.tz_convert(tz)
else:
    # try to parse the ISO string column if present
    if 'earliest_anomaly_timestamp_ASIA/Shanghai' in ga.columns:
        ga['earliest_anomaly_dt'] = pd.to_datetime(ga['earliest_anomaly_timestamp_ASIA/Shanghai'], utc=True).dt.tz_convert(tz)
    else:
        # create a column with NaT if no timestamp available
        ga['earliest_anomaly_dt'] = pd.NaT

nodes = [f"node-{i}" for i in range(1,7)]

rows = []
for node in nodes:
    # select rows where kpi_name contains the node substring
    mask = ga['kpi_name'].astype(str).str.contains(node, na=False)
    node_df = ga[mask & (ga['anomaly_count'] > 0)].copy()
    distinct_kpi_count = int(node_df['kpi_name'].astype(str).nunique())
    sum_anomaly_points = int(node_df['anomaly_count'].sum())
    if not node_df.empty:
        earliest_dt = node_df['earliest_anomaly_dt'].min()
        earliest_iso = earliest_dt.isoformat()
        example_cmdb = ", ".join(node_df['cmdb_id'].astype(str).dropna().unique()[:5])
    else:
        earliest_iso = None
        example_cmdb = ""
    rows.append({
        'node_id': node,
        'distinct_anomalous_kpi_count': distinct_kpi_count,
        'sum_anomaly_points': sum_anomaly_points,
        'earliest_anomaly_timestamp_ASIA/Shanghai': earliest_iso,
        'example_affected_cmdb_ids': example_cmdb
    })

node_anom_summary = pd.DataFrame(rows)

# Sort by distinct_anomalous_kpi_count desc
node_anom_summary = node_anom_summary.sort_values(
    by=['distinct_anomalous_kpi_count', 'sum_anomaly_points'],
    ascending=[False, False]
).reset_index(drop=True)

# Keep full result in kernel variable `node_anom_summary` and display compact top 6 (max 6 nodes)
node_anom_summary.head(6)

```
Out[6]:
```
```python
# Summarize node-level anomaly aggregates from existing `node_anom_summary` variable.
# This is a plain-English concise summary displayed as a single string variable.

try:
    nas = node_anom_summary
except NameError:
    raise RuntimeError("Kernel variable `node_anom_summary` not found. Please run the node aggregation step first.")

# Extract rows for node-5 and node-6 (if present) and general status for other nodes
def row_info(node):
    r = nas[nas['node_id'] == node]
    if r.empty:
        return None
    r = r.iloc[0]
    return {
        'node_id': r['node_id'],
        'distinct_kpis': int(r['distinct_anomalous_kpi_count']),
        'sum_points': int(r['sum_anomaly_points']),
        'earliest': r['earliest_anomaly_timestamp_ASIA/Shanghai'],
        'examples': (r['example_affected_cmdb_ids'] or "")[:200]  # keep examples compact
    }

n5 = row_info('node-5')
n6 = row_info('node-6')

summary_lines = []
summary_lines.append("Node-level anomaly summary (based on metric P95 anomalies):")
if n5:
    summary_lines.append(f"- node-5: {n5['distinct_kpis']} distinct anomalous KPIs, total anomaly points = {n5['sum_points']}, earliest anomaly = {n5['earliest']}. Example affected cmdb_ids: {n5['examples']}.")
else:
    summary_lines.append("- node-5: no anomalies detected.")

if n6:
    summary_lines.append(f"- node-6: {n6['distinct_kpis']} distinct anomalous KPIs, total anomaly points = {n6['sum_points']}, earliest anomaly = {n6['earliest']}. Example affected cmdb_ids: {n6['examples']}.")
else:
    summary_lines.append("- node-6: no anomalies detected.")

# Nodes 1-4 had no anomalous KPIs
summary_lines.append("- node-1 through node-4: no anomalous KPIs detected (distinct_anomalous_kpi_count = 0).")
summary_lines.append("")
summary_lines.append("Interpretation: The anomalies are concentrated on node-5 and node-6 (both show 64 affected KPI types), with node-5 showing a much larger total anomaly point count, suggesting node-5 is the most heavily impacted. Earliest anomalies for affected nodes begin at 2022-03-21T11:00:00+08:00.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

node_id  distinct_anomalous_kpi_count  sum_anomaly_points earliest_anomaly_timestamp_ASIA/Shanghai                          example_affected_cmdb_ids
0  node-5                            64               49283                2022-03-21T11:00:00+08:00  adservice-0, adservice-1, adservice-2, cartser...
1  node-6                            64                9844                2022-03-21T11:00:00+08:00  adservice2-0, cartservice2-0, currencyservice2...
2  node-1                             0                   0                                     None                                                   
3  node-2                             0                   0                                     None                                                   
4  node-3                             0                   0                                     None                                                   
5  node-4                             0                   0                                     None```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse grouped_all from kernel
try:
    ga = grouped_all
except NameError:
    raise RuntimeError("Kernel variable `grouped_all` not found. Please run the metric grouping step first.")

# Ensure necessary columns exist
required_cols = {'cmdb_id', 'kpi_name', 'anomaly_count', 'max_value_in_window'}
missing = required_cols - set(ga.columns)
if missing:
    raise RuntimeError(f"Missing columns in grouped_all: {missing}")

# Ensure numeric anomaly_count
ga['anomaly_count'] = pd.to_numeric(ga['anomaly_count'], errors='coerce').fillna(0).astype(int)

# Normalize earliest anomaly datetime column: prefer existing datetime column or parse ISO string
if 'earliest_anomaly_dt' in ga.columns:
    ga['earliest_anomaly_dt'] = pd.to_datetime(ga['earliest_anomaly_dt'], utc=True).dt.tz_convert(tz)
elif 'earliest_anomaly_timestamp_ASIA/Shanghai' in ga.columns:
    ga['earliest_anomaly_dt'] = pd.to_datetime(ga['earliest_anomaly_timestamp_ASIA/Shanghai'], utc=True).dt.tz_convert(tz)
else:
    ga['earliest_anomaly_dt'] = pd.NaT

candidates = ['node-5','shippingservice2-0','checkoutservice-2']

rows = []
for comp in candidates:
    comp_df = ga[(ga['cmdb_id'] == comp) & (ga['anomaly_count'] > 0)].copy()
    distinct_anomalous_kpi_count = int(comp_df['kpi_name'].nunique())
    sum_anomaly_points = int(comp_df['anomaly_count'].sum())
    if not comp_df.empty:
        earliest_dt = comp_df['earliest_anomaly_dt'].min()
        earliest_iso = earliest_dt.isoformat()
        # prepare top kpis ordered by anomaly_count desc, then max_value_in_window desc
        top_kpis = (
            comp_df.sort_values(by=['anomaly_count', 'max_value_in_window'], ascending=[False, False])
            .loc[:, ['kpi_name', 'anomaly_count', 'max_value_in_window']]
            .drop_duplicates(subset=['kpi_name'])
            .head(5)
        )
        # format each as kpi_name|count|max (keep max value compact)
        def fmt_row(r):
            # format max_value to reasonable precision
            mv = r['max_value_in_window']
            if pd.isna(mv):
                mv_s = "nan"
            else:
                # use 6 significant digits max, drop trailing zeros
                mv_s = ('{:.6g}'.format(float(mv)))
            return f"{r['kpi_name']}|{int(r['anomaly_count'])}|{mv_s}"
        top5_samples = ", ".join(top_kpis.apply(fmt_row, axis=1).tolist())
    else:
        earliest_iso = None
        top5_samples = ""
    rows.append({
        'cmdb_id': comp,
        'distinct_anomalous_kpi_count': distinct_anomalous_kpi_count,
        'sum_anomaly_points': sum_anomaly_points,
        'earliest_anomaly_timestamp_ASIA/Shanghai': earliest_iso,
        'top5_kpi_samples': top5_samples
    })

candidate_component_details = pd.DataFrame(rows)

# Store full result in kernel variable `candidate_component_details` (already assigned)
# Display at most 3 rows (there are 3 candidates)
candidate_component_details.head(3)

```
Out[7]:
```
```python
# Summarize candidate component details in plain English using existing kernel variable
try:
    ccd = candidate_component_details
except NameError:
    raise RuntimeError("Kernel variable `candidate_component_details` not found. Please run the previous step.")

lines = []
lines.append("Candidate component anomaly summary (based on grouped metric anomalies):")

for _, r in ccd.iterrows():
    cm = r['cmdb_id']
    dk = int(r['distinct_anomalous_kpi_count'])
    sp = int(r['sum_anomaly_points'])
    et = r['earliest_anomaly_timestamp_ASIA/Shanghai'] or "N/A"
    top5 = (r['top5_kpi_samples'] or "")
    # keep top5 samples compact (max 200 chars)
    if len(top5) > 200:
        top5 = top5[:197] + "..."
    lines.append(f"- {cm}: {dk} distinct anomalous KPIs, total anomaly points = {sp}, earliest anomaly = {et}. Top KPI samples: {top5}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  distinct_anomalous_kpi_count  sum_anomaly_points earliest_anomaly_timestamp_ASIA/Shanghai                                   top5_kpi_samples
0              node-5                            58                 772                2022-03-21T11:00:00+08:00  system.disk.total|30|6.19142e+08, system.fs.in...
1  shippingservice2-0                            80                1526                2022-03-21T11:00:00+08:00  container.node-5.container_start_time_seconds|...
2   checkoutservice-2                           130                1924                2022-03-21T11:00:00+08:00  container.node-5.container_start_time_seconds|...```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse grouped_all from kernel
try:
    ga = grouped_all
except NameError:
    raise RuntimeError("Kernel variable `grouped_all` not found. Please run the metric grouping step first.")

# Ensure required columns exist
required = {'cmdb_id', 'kpi_name', 'p95_threshold', 'anomaly_count', 'max_value_in_window'}
missing = required - set(ga.columns)
if missing:
    raise RuntimeError(f"Missing required columns in grouped_all: {missing}")

# Normalize numeric columns
ga['p95_threshold'] = pd.to_numeric(ga['p95_threshold'], errors='coerce')
ga['max_value_in_window'] = pd.to_numeric(ga['max_value_in_window'], errors='coerce')
ga['anomaly_count'] = pd.to_numeric(ga['anomaly_count'], errors='coerce').fillna(0).astype(int)

# Normalize earliest anomaly datetime column (create earliest_anomaly_dt if needed)
if 'earliest_anomaly_dt' in ga.columns:
    ga['earliest_anomaly_dt'] = pd.to_datetime(ga['earliest_anomaly_dt'], utc=True).dt.tz_convert(tz)
elif 'earliest_anomaly_timestamp_ASIA/Shanghai' in ga.columns:
    ga['earliest_anomaly_dt'] = pd.to_datetime(ga['earliest_anomaly_timestamp_ASIA/Shanghai'], utc=True).dt.tz_convert(tz)
else:
    ga['earliest_anomaly_dt'] = pd.NaT

candidates = ['node-5','shippingservice2-0','checkoutservice-2']

severity_rows = []
agg_rows = []

for comp in candidates:
    sel = ga[(ga['cmdb_id'] == comp) & (ga['anomaly_count'] > 0)].copy()
    # compute relative_breach safely
    denom = np.maximum(np.abs(sel['p95_threshold']), 1e-9)
    sel['relative_breach'] = (sel['max_value_in_window'] - sel['p95_threshold']) / denom
    # keep relevant columns
    sel = sel.loc[:, ['cmdb_id', 'kpi_name', 'p95_threshold', 'anomaly_count', 'earliest_anomaly_dt', 'earliest_anomaly_timestamp_ASIA/Shanghai', 'max_value_in_window', 'relative_breach']]
    # store full per-candidate KPI-level rows
    severity_rows.append(sel.assign(candidate=comp))
    # aggregates
    distinct_kpis = int(sel['kpi_name'].nunique())
    sum_points = int(sel['anomaly_count'].sum())
    if not sel['earliest_anomaly_dt'].isna().all():
        earliest_dt = sel['earliest_anomaly_dt'].min()
        earliest_iso = earliest_dt.isoformat()
    else:
        earliest_iso = None
    agg_rows.append({
        'cmdb_id': comp,
        'distinct_anomalous_kpi_count': distinct_kpis,
        'sum_anomaly_points': sum_points,
        'earliest_anomaly_timestamp_ASIA/Shanghai': earliest_iso
    })

# Concatenate full per-candidate KPI-level table and store in kernel variable
candidate_kpi_severity = pd.concat(severity_rows, ignore_index=True) if severity_rows else pd.DataFrame(columns=['cmdb_id','kpi_name','p95_threshold','anomaly_count','earliest_anomaly_dt','earliest_anomaly_timestamp_ASIA/Shanghai','max_value_in_window','relative_breach','candidate'])

# Prepare aggregated summary table and store in kernel variable
candidate_kpi_aggregates = pd.DataFrame(agg_rows)

# For display: top 10 kpis per component ordered by relative_breach desc (limit total displayed rows to 30)
display_top = (
    candidate_kpi_severity
    .sort_values(by=['cmdb_id', 'relative_breach'], ascending=[True, False])
    .groupby('cmdb_id', sort=False)
    .head(10)
    .reset_index(drop=True)
)

# Keep only compact columns for display and format earliest timestamp
if 'earliest_anomaly_dt' in display_top.columns:
    display_top['earliest_anomaly_timestamp_ASIA/Shanghai'] = display_top['earliest_anomaly_dt'].apply(lambda x: x.isoformat() if pd.notna(x) else None)

display_cols = ['cmdb_id', 'kpi_name', 'p95_threshold', 'anomaly_count', 'earliest_anomaly_timestamp_ASIA/Shanghai', 'max_value_in_window', 'relative_breach']

# Final displayed outputs (compact)
candidate_kpi_severity, candidate_kpi_aggregates, display_top.loc[:, display_cols].head(30)

```
Out[8]:
```
```python
# Compose a plain-English summary using existing kernel variables:
# candidate_kpi_severity (full per-kpi rows) and candidate_kpi_aggregates (per-component aggregates).
# Display a concise summary string.

try:
    csev = candidate_kpi_severity
    cagg = candidate_kpi_aggregates
except NameError:
    raise RuntimeError("Required kernel variables `candidate_kpi_severity` or `candidate_kpi_aggregates` not found. Run the previous analysis steps first.")

def fmt_val(v):
    try:
        if pd.isna(v):
            return "N/A"
        # format floats nicely
        return f"{float(v):.6g}"
    except Exception:
        return str(v)

import pandas as pd

lines = []
lines.append("Severity summary for candidates: node-5, shippingservice2-0, checkoutservice-2")
lines.append("Data source: per-(cmdb_id,kpi_name) grouped metrics with P95 anomalies (relative breach = (max - p95)/max(|p95|,1e-9)).")
lines.append("")

# Ensure relative_breach present
if 'relative_breach' not in csev.columns:
    # compute if possible
    if {'p95_threshold','max_value_in_window'}.issubset(csev.columns):
        denom = csev['p95_threshold'].abs().clip(lower=1e-9)
        csev['relative_breach'] = (csev['max_value_in_window'] - csev['p95_threshold']) / denom
    else:
        csev['relative_breach'] = 0.0

for _, row in cagg.iterrows():
    comp = row['cmdb_id']
    dk = int(row['distinct_anomalous_kpi_count'])
    sp = int(row['sum_anomaly_points'])
    earliest = row['earliest_anomaly_timestamp_ASIA/Shanghai'] or "N/A"
    lines.append(f"- {comp}: {dk} distinct anomalous KPIs, total anomaly points = {sp}, earliest anomaly = {earliest}.")
    # select top kpis by relative_breach for this component
    comp_kpis = csev[csev['cmdb_id'] == comp].copy()
    if comp_kpis.empty:
        lines.append("  Top KPIs: none with anomaly_count>0.")
        continue
    topk = (comp_kpis.sort_values(by='relative_breach', ascending=False)
            .head(10)
            .loc[:, ['kpi_name','anomaly_count','max_value_in_window','p95_threshold','relative_breach']])
    # prepare up to 3 illustrative KPI entries (kpi|count|max|p95|rel)
    top_items = []
    for _, k in topk.head(3).iterrows():
        kname = k['kpi_name']
        cnt = int(k['anomaly_count'])
        mv = fmt_val(k['max_value_in_window'])
        p95 = fmt_val(k['p95_threshold'])
        rel = fmt_val(k['relative_breach'])
        top_items.append(f"{kname} | count={cnt} | max={mv} | p95={p95} | rel={rel}")
    lines.append("  Top KPI examples (by relative breach): " + "; ".join(top_items))
    lines.append("")

lines.append("Interpretation:")
lines.append("- checkoutservice-2 and shippingservice2-0 show many anomalous KPIs; checkoutservice-2 contains very large relative breaches for several mesh/source metrics (e.g., relative breaches >> 0).")
lines.append("- node-5 shows fewer extreme relative breaches overall, but still has notable CPU/iowait and disk KPIs flagged; node-5 also had a large total anomaly point count in earlier aggregation, suggesting broad impact across many KPIs.")
lines.append("- Earliest anomalies for all three candidates start at 2022-03-21T11:00:00+08:00 in the aggregated results.")
lines.append("")
lines.append("Recommendation: Inspect the listed top KPI examples and their raw time series/messages for these components (especially checkoutservice-2 top relative breaches) to validate true root causes versus metrics with trivial P95 values (e.g., P95=0).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                           kpi_name  p95_threshold  anomaly_count       earliest_anomaly_dt earliest_anomaly_timestamp_ASIA/Shanghai  max_value_in_window  relative_breach          candidate
0               node-5                                  system.cpu.iowait   6.000000e-02              5 2022-03-21 11:08:00+08:00                2022-03-21T11:08:00+08:00         8.000000e-02         0.333333             node-5
1               node-5                               system.cpu.pct_usage   7.486000e+00              2 2022-03-21 11:17:00+08:00                2022-03-21T11:17:00+08:00         7.740000e+00         0.033930             node-5
2               node-5                                  system.cpu.system   3.000000e+00              3 2022-03-21 11:23:00+08:00                2022-03-21T11:23:00+08:00         3.080000e+00         0.026667             node-5
3               node-5                                    system.cpu.user   4.629000e+00              2 2022-03-21 11:09:00+08:00                2022-03-21T11:09:00+08:00         4.760000e+00         0.028300             node-5
4               node-5                                   system.disk.free   5.491574e+08              2 2022-03-21 11:00:00+08:00                2022-03-21T11:00:00+08:00         5.491629e+08         0.000010             node-5
..                 ...                                                ...            ...            ...                       ...                                      ...                  ...              ...                ...
263  checkoutservice-2  mesh.destination.frontend.checkoutservice.isti...   0.000000e+00             21 2022-03-21 11:09:00+08:00                2022-03-21T11:09:00+08:00         0.000000e+00         0.000000  checkoutservice-2
264  checkoutservice-2  mesh.source.checkoutservice.productcatalogserv...   0.000000e+00             21 2022-03-21 11:09:00+08:00                2022-03-21T11:09:00+08:00         0.000000e+00         0.000000  checkoutservice-2
265  checkoutservice-2  mesh.source.checkoutservice.productcatalogserv...   0.000000e+00             21 2022-03-21 11:09:00+08:00                2022-03-21T11:09:00+08:00         0.000000e+00         0.000000  checkoutservice-2
266  checkoutservice-2  mesh.source.checkoutservice.productcatalogserv...   0.000000e+00             21 2022-03-21 11:09:00+08:00                2022-03-21T11:09:00+08:00         0.000000e+00         0.000000  checkoutservice-2
267  checkoutservice-2  mesh.source.checkoutservice.productcatalogserv...   0.000000e+00             21 2022-03-21 11:09:00+08:00                2022-03-21T11:09:00+08:00         0.000000e+00         0.000000  checkoutservice-2

[268 rows x 9 columns],               cmdb_id  distinct_anomalous_kpi_count  sum_anomaly_points earliest_anomaly_timestamp_ASIA/Shanghai
0              node-5                            58                 772                2022-03-21T11:00:00+08:00
1  shippingservice2-0                            80                1526                2022-03-21T11:00:00+08:00
2   checkoutservice-2                           130                1924                2022-03-21T11:00:00+08:00,                cmdb_id                                           kpi_name  p95_threshold  anomaly_count earliest_anomaly_timestamp_ASIA/Shanghai  max_value_in_window  relative_breach
0    checkoutservice-2  mesh.source.checkoutservice.shippingservice.is...       0.010500              3                2022-03-21T11:08:00+08:00             1.650000       156.142857
1    checkoutservice-2  mesh.source.checkoutservice.currencyservice.is...       0.956976              2                2022-03-21T11:07:00+08:00             1.729550         0.807307
2    checkoutservice-2  container.node-5.container_network_receive_MB....       0.200319              2                2022-03-21T11:20:00+08:00             0.324219         0.618514
3    checkoutservice-2  mesh.source.checkoutservice.productcatalogserv...       0.139841              2                2022-03-21T11:07:00+08:00             0.212550         0.519938
4    checkoutservice-2  mesh.source.checkoutservice.jaeger-collector.i...      10.453045              2                2022-03-21T11:08:00+08:00            14.733750         0.409518
..                 ...                                                ...            ...            ...                                      ...                  ...              ...
25  shippingservice2-0      container.node-5.container_cpu_system_seconds       0.012750              2                2022-03-21T11:03:00+08:00             0.015000         0.176471
26  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....       0.001325              2                2022-03-21T11:21:00+08:00             0.001550         0.169811
27  shippingservice2-0       container.node-5.container_cpu_usage_seconds       0.031734              2                2022-03-21T11:19:00+08:00             0.035609         0.122126
28  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....      20.325000              2                2022-03-21T11:27:00+08:00            22.000000         0.082411
29  shippingservice2-0         container.node-5.container_cpu_cfs_periods      44.375000              2                2022-03-21T11:19:00+08:00            47.500000         0.070423

[30 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse candidate_kpi_severity from kernel
try:
    candidate_kpi_severity
except NameError:
    raise RuntimeError("Kernel variable `candidate_kpi_severity` not found. Please run previous steps to create it.")

candidates = ['node-5','shippingservice2-0','checkoutservice-2']

# Ensure numeric columns exist
for col in ['p95_threshold', 'max_value_in_window', 'anomaly_count']:
    if col in candidate_kpi_severity.columns:
        candidate_kpi_severity[col] = pd.to_numeric(candidate_kpi_severity[col], errors='coerce')
    else:
        candidate_kpi_severity[col] = np.nan

# Ensure relative_breach exists; compute if missing
if 'relative_breach' not in candidate_kpi_severity.columns:
    denom = candidate_kpi_severity['p95_threshold'].abs().clip(lower=1e-9)
    candidate_kpi_severity['relative_breach'] = (candidate_kpi_severity['max_value_in_window'] - candidate_kpi_severity['p95_threshold']) / denom

# Ensure earliest timestamp ISO column exists
if 'earliest_anomaly_timestamp_ASIA/Shanghai' not in candidate_kpi_severity.columns:
    if 'earliest_anomaly_dt' in candidate_kpi_severity.columns:
        candidate_kpi_severity['earliest_anomaly_timestamp_ASIA/Shanghai'] = candidate_kpi_severity['earliest_anomaly_dt'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    else:
        candidate_kpi_severity['earliest_anomaly_timestamp_ASIA/Shanghai'] = None

# Filter to candidates and anomaly_count > 0 (only meaningful anomalies)
sel = candidate_kpi_severity[
    (candidate_kpi_severity['cmdb_id'].isin(candidates)) &
    (candidate_kpi_severity['anomaly_count'] > 0)
].copy()

# Sort by relative_breach desc and take top 3 per candidate
sel_sorted = sel.sort_values(by=['cmdb_id', 'relative_breach'], ascending=[True, False])
candidate_top_kpis = sel_sorted.groupby('cmdb_id', sort=False).head(3).reset_index(drop=True)

# Select and order requested columns
candidate_top_kpis = candidate_top_kpis.loc[:, [
    'cmdb_id',
    'kpi_name',
    'p95_threshold',
    'max_value_in_window',
    'relative_breach',
    'earliest_anomaly_timestamp_ASIA/Shanghai',
    'anomaly_count'
]]

# Keep the full result in kernel variable `candidate_top_kpis` and display it (<=9 rows)
candidate_top_kpis

```
Out[9]:
```
```python
# Build a concise plain-English summary of the top-3 KPIs per candidate using `candidate_top_kpis`.
# Reuse existing kernel variable candidate_top_kpis.

try:
    df = candidate_top_kpis
except NameError:
    raise RuntimeError("Kernel variable `candidate_top_kpis` not found. Run the previous step.")

lines = []
lines.append("Top-3 KPI spikes per candidate (relative breach = (max - p95)/max(|p95|,1e-9)):")

for comp in df['cmdb_id'].unique():
    sub = df[df['cmdb_id'] == comp]
    lines.append(f"- {comp}:")
    for _, r in sub.iterrows():
        kpi = r['kpi_name']
        p95 = r['p95_threshold']
        mv = r['max_value_in_window']
        rel = r['relative_breach']
        cnt = int(r['anomaly_count'])
        # format numeric values compactly
        def fmt(x):
            try:
                return f"{float(x):.6g}"
            except Exception:
                return str(x)
        lines.append(f"  • {kpi} | count={cnt} | max={fmt(mv)} | p95={fmt(p95)} | rel={fmt(rel)}")
    # brief interpretation per component
    top_rel = sub['relative_breach'].max()
    if top_rel is not None and top_rel > 5:
        lines.append(f"  → Note: {comp} has very large relative breaches (e.g., rel={fmt(top_rel)}), indicating strong spikes above P95.")
    lines.append("")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  p95_threshold  max_value_in_window  relative_breach earliest_anomaly_timestamp_ASIA/Shanghai  anomaly_count
0   checkoutservice-2  mesh.source.checkoutservice.shippingservice.is...       0.010500             1.650000       156.142857                2022-03-21T11:08:00+08:00              3
1   checkoutservice-2  mesh.source.checkoutservice.currencyservice.is...       0.956976             1.729550         0.807307                2022-03-21T11:07:00+08:00              2
2   checkoutservice-2  container.node-5.container_network_receive_MB....       0.200319             0.324219         0.618514                2022-03-21T11:20:00+08:00              2
3              node-5                                     system.io.util       0.832500             1.350000         0.621622                2022-03-21T11:10:00+08:00              2
4              node-5                                    system.io.svctm       0.230500             0.350000         0.518438                2022-03-21T11:10:00+08:00              2
5              node-5                                  system.cpu.iowait       0.060000             0.080000         0.333333                2022-03-21T11:08:00+08:00              5
6  shippingservice2-0  container.node-5.container_network_receive_MB....       0.043251             0.569785        12.173906                2022-03-21T11:00:00+08:00              3
7  shippingservice2-0  mesh.destination.checkoutservice2.shippingserv...       0.000525             0.001050         1.000000                2022-03-21T11:25:00+08:00              3
8  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...       0.001339             0.002100         0.568627                2022-03-21T11:03:00+08:00              2```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variable
try:
    df = candidate_kpi_severity.copy()
except NameError:
    raise RuntimeError("Kernel variable `candidate_kpi_severity` not found. Please run prior steps to create it.")

# Ensure numeric columns
for col in ['p95_threshold', 'max_value_in_window', 'anomaly_count']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        df[col] = np.nan

# Compute relative_breach if missing
if 'relative_breach' not in df.columns or df['relative_breach'].isna().all():
    denom = df['p95_threshold'].abs().fillna(0).clip(lower=1e-9)
    df['relative_breach'] = (df['max_value_in_window'].fillna(0) - df['p95_threshold'].fillna(0)) / denom

# Ensure earliest timestamp column exists in ISO string form
if 'earliest_anomaly_timestamp_ASIA/Shanghai' not in df.columns:
    if 'earliest_anomaly_dt' in df.columns:
        df['earliest_anomaly_timestamp_ASIA/Shanghai'] = pd.to_datetime(df['earliest_anomaly_dt'], utc=True).dt.tz_convert(tz).apply(lambda x: x.isoformat() if pd.notna(x) else None)
    else:
        df['earliest_anomaly_timestamp_ASIA/Shanghai'] = None
else:
    # try to normalize strings to ISO if possible
    try:
        parsed = pd.to_datetime(df['earliest_anomaly_timestamp_ASIA/Shanghai'], utc=True)
        df['earliest_anomaly_timestamp_ASIA/Shanghai'] = parsed.dt.tz_convert(tz).apply(lambda x: x.isoformat() if pd.notna(x) else None)
    except Exception:
        # keep as-is if parsing fails
        pass

# Select required columns and sort by severity (relative_breach desc), tie-breaker by anomaly_count desc
cols = ['cmdb_id', 'kpi_name', 'p95_threshold', 'max_value_in_window', 'relative_breach', 'earliest_anomaly_timestamp_ASIA/Shanghai', 'anomaly_count']
available_cols = [c for c in cols if c in df.columns]
top_severe_kpis = (
    df.loc[:, available_cols]
    .sort_values(by=['relative_breach', 'anomaly_count'], ascending=[False, False])
    .head(5)
    .reset_index(drop=True)
)

# Display the resulting compact table (stored in kernel variable `top_severe_kpis`)
top_severe_kpis

```
Out[10]:
```
```python
# Summarize the top severe KPIs (from kernel variable `top_severe_kpis`) in plain English.
# The summary will be displayed as a single concise string variable `summary`.

try:
    df = top_severe_kpis
except NameError:
    raise RuntimeError("Kernel variable `top_severe_kpis` not found. Run previous steps first.")

def fmt(x):
    try:
        return f"{float(x):.6g}"
    except Exception:
        return str(x)

lines = []
lines.append("Top severe KPI findings (based on relative breach):")

for i, r in df.iterrows():
    cm = r['cmdb_id']
    kpi = r['kpi_name']
    p95 = fmt(r['p95_threshold'])
    mv = fmt(r['max_value_in_window'])
    rel = fmt(r['relative_breach'])
    et = r.get('earliest_anomaly_timestamp_ASIA/Shanghai', 'N/A')
    cnt = int(r.get('anomaly_count', 0))
    # keep KPI name compact
    kpi_short = (kpi[:120] + "...") if isinstance(kpi, str) and len(kpi) > 120 else kpi
    lines.append(f"{i+1}. {cm} | {kpi_short} | anomalies={cnt} | max={mv} | p95={p95} | rel={rel} | earliest={et}")

lines.append("")
lines.append("Plain-English summary:")
lines.append("- The most severe spike is in checkoutservice-2 (mesh.source.checkoutservice.shippingservice...), with a very large relative breach (~156x) — max 1.65 vs p95 0.0105. This strongly suggests a major spike above normal for that mesh/source metric.")
lines.append("- shippingservice2-0 shows high severity on a container network receive metric (relative breach ~12x) and other mesh destination metrics, indicating network/mesh traffic anomalies involving shippingservice2-0 and node-5.")
lines.append("- node-5 also shows notable IO-related anomalies (system.io.util), but with smaller relative breaches compared to the top mesh/network spikes.")
lines.append("- Earliest anomalies for these top items are within the incident window (around 2022-03-21T11:00:00+08:00 to 11:10+08:00).")
lines.append("")
lines.append("Actionable next steps: prioritize investigating checkoutservice-2 and shippingservice2-0 (and the associated node-5) focusing on mesh/network traces and raw time series for the listed KPIs to determine root cause and whether the P95=0 or tiny P95 values are inflating relative_breach metrics.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  p95_threshold  max_value_in_window  relative_breach earliest_anomaly_timestamp_ASIA/Shanghai  anomaly_count
0   checkoutservice-2  mesh.source.checkoutservice.shippingservice.is...       0.010500             1.650000       156.142857                2022-03-21T11:08:00+08:00              3
1  shippingservice2-0  container.node-5.container_network_receive_MB....       0.043251             0.569785        12.173906                2022-03-21T11:00:00+08:00              3
2  shippingservice2-0  mesh.destination.checkoutservice2.shippingserv...       0.000525             0.001050         1.000000                2022-03-21T11:25:00+08:00              3
3   checkoutservice-2  mesh.source.checkoutservice.currencyservice.is...       0.956976             1.729550         0.807307                2022-03-21T11:07:00+08:00              2
4              node-5                                     system.io.util       0.832500             1.350000         0.621622                2022-03-21T11:10:00+08:00              2```
```